# Kuhn Poker and Counterfactual Regret Minimization Overview



## Imports and Notebook Setup

This section imports the small set of Python tools needed for the notebook and sets a random seed for reproducibility.

In [3]:
import random
from collections import defaultdict

# Reproducibility
random.seed(42)

## Kuhn Poker Rules and State Representation

In [4]:
CARDS = ["J", "Q", "K"]
ACTIONS = ["c", "b"]

def is_terminal(history):
    return history in ["cc", "bc", "bf", "cbc", "cbf"]

def next_player(history):
    return len(history) % 2

def deal_cards():
    deck = CARDS.copy()
    random.shuffle(deck)
    return deck[0], deck[1]

## Information Sets and Node Data Structure

In [5]:
class Node:
    def __init__(self, info_set):
        self.info_set = info_set
        self.actions = ["c", "b"]
        self.num_actions = len(self.actions)

        # CFR data
        self.regret_sum = {a: 0.0 for a in self.actions}
        self.strategy_sum = {a: 0.0 for a in self.actions}

    def get_strategy(self, realization_weight):
        """
        Compute the current mixed strategy using regret matching.
        Also accumulate the weighted strategy for averaging later.
        """
        strategy = {}
        normalizing_sum = 0.0

        for action in self.actions:
            strategy[action] = max(self.regret_sum[action], 0.0)
            normalizing_sum += strategy[action]

        if normalizing_sum > 0:
            for action in self.actions:
                strategy[action] /= normalizing_sum
        else:
            for action in self.actions:
                strategy[action] = 1.0 / self.num_actions

        for action in self.actions:
            self.strategy_sum[action] += realization_weight * strategy[action]

        return strategy

    def get_average_strategy(self):
        """
        Return the average strategy across training.
        """
        avg_strategy = {}
        normalizing_sum = sum(self.strategy_sum.values())

        if normalizing_sum > 0:
            for action in self.actions:
                avg_strategy[action] = self.strategy_sum[action] / normalizing_sum
        else:
            for action in self.actions:
                avg_strategy[action] = 1.0 / self.num_actions

        return avg_strategy

    def __str__(self):
        avg = self.get_average_strategy()
        parts = [f"{a}: {avg[a]:.3f}" for a in self.actions]
        return f"{self.info_set} -> " + ", ".join(parts)
    


In [6]:
# Dictionary mapping information set strings to Node objects
node_map = {}

## Regret Matching

CFR chooses actions using regret matching.


In [7]:
def regret_matching_example(regret_sum):
    """
    Simple standalone regret matching example.

    Input:
        regret_sum: dict like {"c": ..., "b": ...}

    Output:
        strategy: dict of action probabilities
    """
    actions = list(regret_sum.keys())
    strategy = {}
    normalizing_sum = 0.0

    for action in actions:
        strategy[action] = max(regret_sum[action], 0.0)
        normalizing_sum += strategy[action]

    if normalizing_sum > 0:
        for action in actions:
            strategy[action] /= normalizing_sum
    else:
        for action in actions:
            strategy[action] = 1.0 / len(actions)

    return strategy

In [8]:
example_regrets = {"c": 3.0, "b": 1.0}
print(regret_matching_example(example_regrets))

{'c': 0.75, 'b': 0.25}


In [9]:
example_regrets = {"c": -2.0, "b": -5.0}
print(regret_matching_example(example_regrets))

{'c': 0.5, 'b': 0.5}


## Terminal Utility Function

This section computes the payoff once a terminal history is reached.

In [10]:
def card_rank(card):
    """
    Convert card to rank for comparisons.
    """
    ranks = {"J": 0, "Q": 1, "K": 2}
    return ranks[card]

In [11]:
def terminal_utility(history, cards):
    """
    Return utility for Player 0 at a terminal history.

    Parameters
    ----------
    history : str
        Betting history, e.g. "cc", "bf", "cbc"
    cards : tuple
        (player_0_card, player_1_card)

    Returns
    -------
    float
        Payoff from Player 0's perspective.
    """
    player_0_card, player_1_card = cards

    # Showdown after both players stayed in
    if history in ["cc", "bc", "cbc"]:
        if card_rank(player_0_card) > card_rank(player_1_card):
            if history == "cc":
                return 1.0   # wins pot of 2, net +1
            else:
                return 2.0   # wins pot of 4, net +2
        else:
            if history == "cc":
                return -1.0
            else:
                return -2.0

    # Someone folded
    if history == "bf":
        return 1.0   # Player 0 bet, Player 1 folded

    if history == "cbf":
        return -1.0  # Player 1 bet, Player 0 folded

    raise ValueError(f"Non-terminal or unknown history: {history}")

In [12]:
print(terminal_utility("cc", ("K", "J")))    # expected +1
print(terminal_utility("cc", ("J", "K")))    # expected -1
print(terminal_utility("bf", ("J", "K")))    # expected +1
print(terminal_utility("cbf", ("K", "J")))   # expected -1
print(terminal_utility("bc", ("K", "Q")))    # expected +2
print(terminal_utility("cbc", ("J", "Q")))   # expected -2

1.0
-1.0
1.0
-1.0
2.0
-2.0
